In [1]:
import time
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("garments_worker_productivity.csv")

df["department"] = df["department"].str.strip()
df["department"] = df["department"].replace("sweing", "sewing")
df = df.drop(columns=["date"])
df["MeetsTarget"] = (df["actual_productivity"] >= df["targeted_productivity"]).astype(int)

train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)
train_df = train_df.sort_index()
test_df = test_df.sort_index()

cat_cols = ["quarter", "department", "day", "team"]
num_cols = ["targeted_productivity", "smv", "wip", "over_time",
            "incentive", "idle_time", "idle_men",
            "no_of_style_change", "no_of_workers"]

wip_median = train_df["wip"].median()
train_df = train_df.copy()
test_df = test_df.copy()
train_df["wip"] = train_df["wip"].fillna(wip_median)
test_df["wip"] = test_df["wip"].fillna(wip_median)

train_dummies = pd.get_dummies(train_df[cat_cols], columns=cat_cols)
test_dummies = pd.get_dummies(test_df[cat_cols], columns=cat_cols)
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)
train_dummies = train_dummies.astype(float)
test_dummies = test_dummies.astype(float)

num_mean = train_df[num_cols].mean()
num_std = train_df[num_cols].std(ddof=0)
train_num_scaled = (train_df[num_cols] - num_mean) / num_std
test_num_scaled = (test_df[num_cols] - num_mean) / num_std

train_X = pd.concat([train_num_scaled, train_dummies], axis=1)
test_X = pd.concat([test_num_scaled, test_dummies], axis=1)

print("train_X shape:", train_X.shape)
print("test_X shape:", test_X.shape)

train_X shape: (958, 34)
test_X shape: (239, 34)


In [3]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def logistic_fit_newton(X, y, l2=1e-6, max_iter=50, tol=1e-8):
    n, d = X.shape
    w = np.zeros(d)
    prev_loss = np.inf
    for it in range(max_iter):
        z = X @ w
        p = sigmoid(z)

        # gradient and Hessian with small L2
        grad = (X.T @ (p - y)) / n
        grad[1:] += (l2 / n) * w[1:]

        W = p * (1 - p)
        H = (X.T * W) @ X / n
        H[1:, 1:] += (l2 / n) * np.eye(d - 1)

        # Newton step
        try:
            step = np.linalg.solve(H, grad)
        except np.linalg.LinAlgError:
            break
        w = w - step

        eps = 1e-12
        loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
        if abs(prev_loss - loss) < tol:
            break
        prev_loss = loss
    return w


def logistic_fit_gd(X, y, lr=0.1, epochs=2000, l2=0.0):
    n, d = X.shape
    w = np.zeros(d)
    for _ in range(epochs):
        p = sigmoid(X @ w)
        grad = (X.T @ (p - y)) / n
        if l2 > 0:
            grad[1:] += (l2 / n) * w[1:]
        w -= lr * grad
    return w

In [4]:
X_train_clf = train_X.values
y_train_clf = train_df["MeetsTarget"].values
X_test_clf = test_X.values
y_test_clf = test_df["MeetsTarget"].values

X_train_clf_b = np.hstack([np.ones((X_train_clf.shape[0], 1)), X_train_clf])
X_test_clf_b = np.hstack([np.ones((X_test_clf.shape[0], 1)), X_test_clf])


def evaluate_classifier(w, X_test_b, y_test):
    probs = sigmoid(X_test_b @ w)
    preds = (probs >= 0.5).astype(int)
    tp = np.sum((preds == 1) & (y_test == 1))
    fp = np.sum((preds == 1) & (y_test == 0))
    fn = np.sum((preds == 0) & (y_test == 1))
    tn = np.sum((preds == 0) & (y_test == 0))
    acc = (tp + tn) / len(y_test)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return acc, prec, rec, f1


# old approach (Part B)
t0 = time.perf_counter()
w_gd = logistic_fit_gd(X_train_clf_b, y_train_clf, lr=0.1, epochs=2000)
gd_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
gd_metrics = evaluate_classifier(w_gd, X_test_clf_b, y_test_clf)
gd_pred_time = time.perf_counter() - t0

# new approach (Newton-Raphson / IRLS)
t0 = time.perf_counter()
w_newton = logistic_fit_newton(X_train_clf_b, y_train_clf, l2=1e-6)
newton_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
newton_metrics = evaluate_classifier(w_newton, X_test_clf_b, y_test_clf)
newton_pred_time = time.perf_counter() - t0

print("Logistic Regression: gradient descent (Part B) vs Newton-Raphson (Part C)")
print()
print("{:<28}{:>14}{:>14}".format("", "GD (Part B)", "Newton (Part C)"))
print("{:<28}{:>14.6f}{:>14.6f}".format("train time (s)", gd_train_time, newton_train_time))
print("{:<28}{:>14.6f}{:>14.6f}".format("predict time (s)", gd_pred_time, newton_pred_time))
print("{:<28}{:>14.6f}{:>14.6f}".format("accuracy", gd_metrics[0], newton_metrics[0]))
print("{:<28}{:>14.6f}{:>14.6f}".format("precision", gd_metrics[1], newton_metrics[1]))
print("{:<28}{:>14.6f}{:>14.6f}".format("recall", gd_metrics[2], newton_metrics[2]))
print("{:<28}{:>14.6f}{:>14.6f}".format("f1", gd_metrics[3], newton_metrics[3]))

Logistic Regression: gradient descent (Part B) vs Newton-Raphson (Part C)

                               GD (Part B)Newton (Part C)
train time (s)                    0.048376      0.001983
predict time (s)                  0.000076      0.000049
accuracy                          0.757322      0.753138
precision                         0.783920      0.785714
recall                            0.912281      0.900585
f1                                0.843243      0.839237


In [5]:
X_train_reg = train_X.values
y_train_reg = train_df["actual_productivity"].values
X_test_reg = test_X.values
y_test_reg = test_df["actual_productivity"].values

X_train_reg_b = np.hstack([np.ones((X_train_reg.shape[0], 1)), X_train_reg])
X_test_reg_b = np.hstack([np.ones((X_test_reg.shape[0], 1)), X_test_reg])


def linear_fit_ridge(X, y, l2=1e-6):
    n, d = X.shape
    I = np.eye(d)
    I[0, 0] = 0  # don't regularize the intercept
    A = X.T @ X + l2 * I
    b = X.T @ y
    return np.linalg.solve(A, b)


t0 = time.perf_counter()
w_ridge = linear_fit_ridge(X_train_reg_b, y_train_reg, l2=1e-6)
ridge_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_ridge = X_test_reg_b @ w_ridge
ridge_pred_time = time.perf_counter() - t0

mae = np.mean(np.abs(y_test_reg - y_pred_ridge))
rmse = np.sqrt(np.mean((y_test_reg - y_pred_ridge) ** 2))
ss_res = np.sum((y_test_reg - y_pred_ridge) ** 2)
ss_tot = np.sum((y_test_reg - np.mean(y_test_reg)) ** 2)
r2 = 1 - ss_res / ss_tot

print("Linear Regression (Part C, ridge closed-form)")
print("train time:   {:.6f} s".format(ridge_train_time))
print("predict time: {:.6f} s".format(ridge_pred_time))
print("MAE:  {:.6f}".format(mae))
print("RMSE: {:.6f}".format(rmse))
print("R2:   {:.6f}".format(r2))

Linear Regression (Part C, ridge closed-form)
train time:   0.001459 s
predict time: 0.000038 s
MAE:  0.101798
RMSE: 0.141897
R2:   0.348388


In [6]:
# values from earlier parts, reproduced here for a single reference table
sklearn_lr =  {"train": 0.003864, "predict": 0.001017, "mae": 0.101798, "rmse": 0.141897, "r2": 0.348388}
manual_lr =   {"train": 0.005915, "predict": 0.000064, "mae": 0.101798, "rmse": 0.141897, "r2": 0.348388}
optim_lr =    {"train": ridge_train_time, "predict": ridge_pred_time, "mae": mae, "rmse": rmse, "r2": r2}

sklearn_log = {"train": 0.009557, "predict": 0.000140, "acc": 0.761506, "prec": 0.787879, "rec": 0.912281, "f1": 0.845528}
manual_log =  {"train": gd_train_time, "predict": gd_pred_time, "acc": gd_metrics[0], "prec": gd_metrics[1], "rec": gd_metrics[2], "f1": gd_metrics[3]}
optim_log =   {"train": newton_train_time, "predict": newton_pred_time, "acc": newton_metrics[0], "prec": newton_metrics[1], "rec": newton_metrics[2], "f1": newton_metrics[3]}

print("Linear Regression")
print("{:<18}{:>14}{:>14}{:>14}".format("", "sklearn", "manual", "optimized"))
for k in ["train", "predict", "mae", "rmse", "r2"]:
    print("{:<18}{:>14.6f}{:>14.6f}{:>14.6f}".format(k, sklearn_lr[k], manual_lr[k], optim_lr[k]))

print()
print("Logistic Regression")
print("{:<18}{:>14}{:>14}{:>14}".format("", "sklearn", "manual", "optimized"))
for k in ["train", "predict", "acc", "prec", "rec", "f1"]:
    print("{:<18}{:>14.6f}{:>14.6f}{:>14.6f}".format(k, sklearn_log[k], manual_log[k], optim_log[k]))

Linear Regression
                         sklearn        manual     optimized
train                   0.003864      0.005915      0.001459
predict                 0.001017      0.000064      0.000038
mae                     0.101798      0.101798      0.101798
rmse                    0.141897      0.141897      0.141897
r2                      0.348388      0.348388      0.348388

Logistic Regression
                         sklearn        manual     optimized
train                   0.009557      0.048376      0.001983
predict                 0.000140      0.000076      0.000049
acc                     0.761506      0.757322      0.753138
prec                    0.787879      0.783920      0.785714
rec                     0.912281      0.912281      0.900585
f1                      0.845528      0.843243      0.839237
